# Model 2: Duration Band Prediction

Predict `duration_band` in the fixed order: `short`, `medium`, `long`, `very_long`.


## 1. Setup

Import dependencies and resolve project paths.


In [180]:
from pathlib import Path
from itertools import product
import warnings

import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'outputs').exists()), cwd)
DATA_PATH = PROJECT_ROOT / 'outputs' / 'model_road_closure' / 'model2_duration_handoff.csv'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'


## 2. Data Contract

Load the handoff and validate the target schema.


In [181]:
df = pd.read_csv(DATA_PATH, low_memory=False)

target_col = 'duration_band'
CLASS_ORDER = ['short', 'medium', 'long', 'very_long']
CLASS_TO_INDEX = {label: idx for idx, label in enumerate(CLASS_ORDER)}
INDEX_TO_CLASS = {idx: label for label, idx in CLASS_TO_INDEX.items()}

assert target_col in df.columns, f'Missing target column: {target_col}'
assert not df.columns.duplicated().any(), 'Duplicate input columns are not allowed.'


## 3. Features

Build chronological, leakage-safe model inputs.


In [182]:
model_df = df.dropna(subset=[target_col]).copy()
model_df[target_col] = model_df[target_col].astype(str).str.lower().str.strip()

unexpected_labels = sorted(set(model_df[target_col]) - set(CLASS_ORDER))
assert not unexpected_labels, f'Unexpected duration_band labels: {unexpected_labels}'
assert set(CLASS_ORDER).issubset(set(model_df[target_col])), 'All four target classes must be present.'

for col in ['prediction_datetime', 'start_datetime']:
    if col in model_df.columns:
        model_df[col] = pd.to_datetime(model_df[col], errors='coerce', utc=True)

sort_cols = [c for c in ['prediction_datetime', 'start_datetime', '_source_row'] if c in model_df.columns]
if sort_cols:
    model_df = model_df.sort_values(sort_cols, kind='mergesort', na_position='last').reset_index(drop=True)

explicit_non_features = {
    target_col,
    'id',
    '_source_row',
    'duration_minutes',
    'duration_minutes_raw',
    'valid_duration_label',
    'target_road_closure',
    'predicted_duration_band',
    'actual_duration_band',
}

def is_non_feature(column_name):
    name = column_name.lower()
    is_timestamp = ('datetime' in name) or ('timestamp' in name)
    is_post_event = name.startswith(('actual_', 'post_event_', 'resolved_', 'reopened_'))
    is_other_target = name.startswith('target_')
    return column_name in explicit_non_features or is_timestamp or is_post_event or is_other_target

excluded_model_columns = [c for c in model_df.columns if is_non_feature(c)]
feature_cols = [c for c in model_df.columns if c not in excluded_model_columns]

X = model_df[feature_cols].copy()
y_raw = model_df[target_col]
y = y_raw.map(CLASS_TO_INDEX).astype(int).to_numpy()

numeric_cols = X.select_dtypes(include=[np.number, 'bool']).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

# Only numeric columns receive numeric coercion; categorical values stay categorical.
X.loc[:, numeric_cols] = X.loc[:, numeric_cols].replace([np.inf, -np.inf], np.nan)

assert not ({'id', '_source_row', target_col} & set(feature_cols))
assert not any(('datetime' in c.lower() or 'timestamp' in c.lower()) for c in feature_cols)


## 4. Split

Create the 70/15/15 chronological train, validation, and test sets.


In [183]:
RANDOM_STATE = 42
SPLIT_METHOD = 'chronological_train_validation_test_split'

n_rows = len(X)
train_end = int(0.70 * n_rows)
val_end = int(0.85 * n_rows)

X_train = X.iloc[:train_end].copy()
X_val = X.iloc[train_end:val_end].copy()
X_test = X.iloc[val_end:].copy()

y_train = y[:train_end].copy()
y_val = y[train_end:val_end].copy()
y_test = y[val_end:].copy()

for split_name, split_y in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    assert set(np.unique(split_y)) == set(range(len(CLASS_ORDER))), f'{split_name} split is missing a class.'


## 5. Modeling Utilities

Define preprocessing, estimators, probability alignment, and metrics.


In [184]:
MODEL_NAMES = ['CatBoost', 'RandomForest', 'XGBoost']
WEIGHT_STRATEGIES = ['none', 'balanced', 'sqrt_balanced']
N_CLASSES = len(CLASS_ORDER)

def class_weight_values(y_fit, strategy):
    counts = np.bincount(np.asarray(y_fit, dtype=int), minlength=N_CLASSES).astype(float)
    balanced = len(y_fit) / (N_CLASSES * counts)
    if strategy == 'none':
        values = np.ones(N_CLASSES, dtype=float)
    elif strategy == 'balanced':
        values = balanced
    elif strategy == 'sqrt_balanced':
        values = np.sqrt(balanced)
    else:
        raise ValueError(f'Unknown class-weight strategy: {strategy}')
    return values

def make_preprocessor(model_name):
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
    ])
    categorical_imputer = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
    ])

    if model_name == 'CatBoost':
        processor = ColumnTransformer(
            transformers=[
                ('numeric', numeric_pipe, numeric_cols),
                ('categorical', categorical_imputer, categorical_cols),
            ],
            remainder='drop',
            verbose_feature_names_out=False,
        )
        return processor.set_output(transform='pandas')

    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32)),
    ])
    return ColumnTransformer(
        transformers=[
            ('numeric', numeric_pipe, numeric_cols),
            ('categorical', categorical_pipe, categorical_cols),
        ],
        remainder='drop',
        sparse_threshold=0.3,
        verbose_feature_names_out=True,
    )

def make_base_estimator(model_name, strategy, y_reference):
    weights = class_weight_values(y_reference, strategy)
    weight_dict = {idx: float(weight) for idx, weight in enumerate(weights)}

    if model_name == 'CatBoost':
        return CatBoostClassifier(
            loss_function='MultiClass',
            eval_metric='TotalF1',
            iterations=300,
            learning_rate=0.04,
            depth=5,
            l2_leaf_reg=5,
            random_seed=RANDOM_STATE,
            class_weights=None if strategy == 'none' else weights.tolist(),
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
            cat_features=list(range(len(numeric_cols), len(numeric_cols) + len(categorical_cols))),
        )
    if model_name == 'RandomForest':
        return RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            class_weight=None if strategy == 'none' else weight_dict,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    if model_name == 'XGBoost':
        return XGBClassifier(
            objective='multi:softprob',
            num_class=N_CLASSES,
            eval_metric='mlogloss',
            n_estimators=300,
            max_depth=4,
            learning_rate=0.035,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_weight=2,
            reg_lambda=2.0,
            reg_alpha=0.1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method='hist',
        )
    raise ValueError(model_name)

def fit_model_pipeline(model_name, strategy, X_fit, y_fit):
    pipeline = Pipeline([
        ('preprocessor', make_preprocessor(model_name)),
        ('model', make_base_estimator(model_name, strategy, y_fit)),
    ])
    fit_kwargs = {}
    if model_name == 'XGBoost' and strategy != 'none':
        fit_kwargs['model__sample_weight'] = class_weight_values(y_fit, strategy)[y_fit]
    pipeline.fit(X_fit, y_fit, **fit_kwargs)
    return pipeline

def align_probability_columns(probabilities, estimator_classes):
    aligned = np.zeros((len(probabilities), N_CLASSES), dtype=float)
    for source_index, class_code in enumerate(np.asarray(estimator_classes, dtype=int)):
        aligned[:, class_code] = probabilities[:, source_index]
    row_sums = aligned.sum(axis=1, keepdims=True)
    return aligned / np.where(row_sums == 0, 1.0, row_sums)

def predict_component_proba(component, X_input):
    if isinstance(component, dict) and component.get('calibrated', False):
        transformed = component['preprocessor'].transform(X_input)
        raw = component['model'].predict_proba(transformed)
        classes = component['model'].classes_
    else:
        raw = component.predict_proba(X_input)
        classes = component.classes_
    return align_probability_columns(raw, classes)

def evaluate_probabilities(y_true, probabilities, predictions=None):
    if predictions is None:
        predictions = probabilities.argmax(axis=1)
    cm = confusion_matrix(y_true, predictions, labels=np.arange(N_CLASSES))
    precision, recall, class_f1, support = precision_recall_fscore_support(
        y_true,
        predictions,
        labels=np.arange(N_CLASSES),
        zero_division=0,
    )
    fp = cm.sum(axis=0) - np.diag(cm)
    fn = cm.sum(axis=1) - np.diag(cm)
    per_class = pd.DataFrame({
        'class': CLASS_ORDER,
        'precision': precision,
        'recall': recall,
        'f1': class_f1,
        'FP': fp.astype(int),
        'FN': fn.astype(int),
        'support': support.astype(int),
    })
    return {
        'accuracy': float(accuracy_score(y_true, predictions)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, predictions)),
        'macro_f1': float(f1_score(y_true, predictions, average='macro')),
        'weighted_f1': float(f1_score(y_true, predictions, average='weighted')),
        'log_loss': float(log_loss(y_true, probabilities, labels=np.arange(N_CLASSES))),
        'per_class': per_class,
        'confusion_matrix': pd.DataFrame(cm, index=CLASS_ORDER, columns=CLASS_ORDER),
        'predictions': np.asarray(predictions, dtype=int),
    }

def compact_metric_row(configuration, details):
    return {
        'configuration': configuration,
        'accuracy': details['accuracy'],
        'balanced_accuracy': details['balanced_accuracy'],
        'macro_f1': details['macro_f1'],
        'weighted_f1': details['weighted_f1'],
    }


## 6. Candidate Training

Train three models under all required class-weight strategies.


In [185]:
candidate_models = {}
candidate_validation = {}
for model_name in MODEL_NAMES:
    for strategy in WEIGHT_STRATEGIES:
        key = (model_name, strategy)
        fitted = fit_model_pipeline(model_name, strategy, X_train, y_train)
        probabilities = predict_component_proba(fitted, X_val)
        candidate_models[key] = fitted
        candidate_validation[key] = {
            'probabilities': probabilities,
            'metrics': evaluate_probabilities(y_val, probabilities),
        }

best_keys = {}
for model_name in MODEL_NAMES:
    model_keys = [(model_name, strategy) for strategy in WEIGHT_STRATEGIES]
    best_keys[model_name] = max(
        model_keys,
        key=lambda key: (
            candidate_validation[key]['metrics']['macro_f1'],
            candidate_validation[key]['metrics']['balanced_accuracy'],
        ),
    )

best_models = {name: candidate_models[best_keys[name]] for name in MODEL_NAMES}
best_strategies = {name: best_keys[name][1] for name in MODEL_NAMES}
best_val_probabilities = {
    name: candidate_validation[best_keys[name]]['probabilities'] for name in MODEL_NAMES
}
best_val_metrics = {
    name: candidate_validation[best_keys[name]]['metrics'] for name in MODEL_NAMES
}


## 7. Ensemble and Calibration

Select soft-voting weights and compare train-only sigmoid calibration.


In [186]:
def blend_probabilities(probability_map, weights):
    blended = sum(weights[name] * probability_map[name] for name in MODEL_NAMES)
    return blended / blended.sum(axis=1, keepdims=True)

initial_weights = {'CatBoost': 0.45, 'RandomForest': 0.35, 'XGBoost': 0.20}
weight_candidates = []
for cat_steps in range(21):
    for rf_steps in range(21 - cat_steps):
        xgb_steps = 20 - cat_steps - rf_steps
        weights = {
            'CatBoost': cat_steps / 20,
            'RandomForest': rf_steps / 20,
            'XGBoost': xgb_steps / 20,
        }
        probabilities = blend_probabilities(best_val_probabilities, weights)
        details = evaluate_probabilities(y_val, probabilities)
        distance_from_start = sum(abs(weights[name] - initial_weights[name]) for name in MODEL_NAMES)
        weight_candidates.append((details['macro_f1'], details['balanced_accuracy'], -distance_from_start, weights))

_, _, _, selected_ensemble_weights = max(weight_candidates)
uncalibrated_ensemble_val = blend_probabilities(best_val_probabilities, selected_ensemble_weights)
uncalibrated_ensemble_metrics = evaluate_probabilities(y_val, uncalibrated_ensemble_val)

def fit_calibrated_component(model_name, strategy, X_fit, y_fit):
    preprocessor = make_preprocessor(model_name)
    transformed = preprocessor.fit_transform(X_fit, y_fit)
    base_model = make_base_estimator(model_name, strategy, y_fit)
    calibrator = CalibratedClassifierCV(
        estimator=base_model,
        method='sigmoid',
        cv=3,
        ensemble=True,
    )
    fit_kwargs = {}
    if model_name == 'XGBoost' and strategy != 'none':
        fit_kwargs['sample_weight'] = class_weight_values(y_fit, strategy)[y_fit]
    calibrator.fit(transformed, y_fit, **fit_kwargs)
    return {'calibrated': True, 'preprocessor': preprocessor, 'model': calibrator}

calibrated_components = {}
calibrated_val_probabilities = dict(best_val_probabilities)
for model_name in MODEL_NAMES:
    # CatBoost with native cat_features is not sklearn-clone-safe; keep its
    # probability stream uncalibrated and calibrate the clone-safe members.
    if selected_ensemble_weights[model_name] > 0 and model_name != 'CatBoost':
        component = fit_calibrated_component(
            model_name,
            best_strategies[model_name],
            X_train,
            y_train,
        )
        calibrated_components[model_name] = component
        calibrated_val_probabilities[model_name] = predict_component_proba(component, X_val)

calibrated_ensemble_val = blend_probabilities(calibrated_val_probabilities, selected_ensemble_weights)
calibrated_ensemble_metrics = evaluate_probabilities(y_val, calibrated_ensemble_val)

calibration_retained_for_ensemble = (
    calibrated_ensemble_metrics['macro_f1'] > uncalibrated_ensemble_metrics['macro_f1'] + 1e-12
    or calibrated_ensemble_metrics['log_loss'] < uncalibrated_ensemble_metrics['log_loss'] - 1e-12
)
if calibration_retained_for_ensemble:
    ensemble_val_probabilities = calibrated_ensemble_val
    ensemble_val_component_probabilities = calibrated_val_probabilities
else:
    ensemble_val_probabilities = uncalibrated_ensemble_val
    ensemble_val_component_probabilities = best_val_probabilities

unadjusted_ensemble_metrics = evaluate_probabilities(y_val, ensemble_val_probabilities)


## 8. Probability Adjustment

Tune class multipliers and the `very_long` threshold on validation only.


In [187]:
def quick_confusion_scores(y_true, predictions):
    cm = np.bincount(
        np.asarray(y_true, dtype=int) * N_CLASSES + np.asarray(predictions, dtype=int),
        minlength=N_CLASSES * N_CLASSES,
    ).reshape(N_CLASSES, N_CLASSES)
    tp = np.diag(cm).astype(float)
    row_sum = cm.sum(axis=1).astype(float)
    col_sum = cm.sum(axis=0).astype(float)
    recall = np.divide(tp, row_sum, out=np.zeros_like(tp), where=row_sum > 0)
    precision = np.divide(tp, col_sum, out=np.zeros_like(tp), where=col_sum > 0)
    class_f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )
    return {
        'macro_f1': float(class_f1.mean()),
        'balanced_accuracy': float(recall.mean()),
        'recall': recall,
        'fp': col_sum - tp,
        'fn': row_sum - tp,
    }

baseline_quick = quick_confusion_scores(y_val, ensemble_val_probabilities.argmax(axis=1))
validation_support = np.bincount(y_val, minlength=N_CLASSES)
recall_tolerance = 1.0 / np.maximum(validation_support, 1)
minimum_recall = np.maximum(0.10, 0.60 * baseline_quick['recall'])
preserve_indices = [CLASS_TO_INDEX[name] for name in ['short', 'medium', 'long']]

multiplier_grid = np.round(np.arange(0.60, 1.401, 0.05), 2)
multiplier_candidates = []
for values in product(multiplier_grid, repeat=N_CLASSES):
    values_array = np.asarray(values, dtype=float)
    predictions = (ensemble_val_probabilities * values_array).argmax(axis=1)
    quick = quick_confusion_scores(y_val, predictions)
    preserves_priority_recall = all(
        quick['recall'][idx] + recall_tolerance[idx] >= baseline_quick['recall'][idx]
        for idx in preserve_indices
    )
    avoids_recall_collapse = bool(np.all(quick['recall'] + recall_tolerance >= minimum_recall))
    if preserves_priority_recall and avoids_recall_collapse:
        multiplier_candidates.append(
            (quick['macro_f1'], quick['balanced_accuracy'], -quick['fp'][CLASS_TO_INDEX['very_long']], values_array)
        )

if not multiplier_candidates:
    multiplier_candidates = [(baseline_quick['macro_f1'], baseline_quick['balanced_accuracy'], -baseline_quick['fp'][3], np.ones(N_CLASSES))]
multiplier_candidates.sort(key=lambda item: item[:3], reverse=True)

very_long_index = CLASS_TO_INDEX['very_long']
baseline_vl_fp = int(baseline_quick['fp'][very_long_index])
required_fp_reduction = 0 if baseline_vl_fp == 0 else max(1, int(np.ceil(0.15 * baseline_vl_fp)))
target_vl_fp = baseline_vl_fp - required_fp_reduction
threshold_grid = np.round(np.arange(0.35, 0.801, 0.05), 2)

adjusted_candidates = []
relaxed_adjusted_candidates = []
for _, _, _, multiplier_values in multiplier_candidates[:5000]:
    adjusted = ensemble_val_probabilities * multiplier_values
    adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)
    for threshold in threshold_grid:
        predictions = adjusted.argmax(axis=1)
        replace_mask = (predictions == very_long_index) & (adjusted[:, very_long_index] < threshold)
        predictions[replace_mask] = adjusted[replace_mask, :very_long_index].argmax(axis=1)
        quick = quick_confusion_scores(y_val, predictions)
        preserves_priority_recall = all(
            quick['recall'][idx] + recall_tolerance[idx] >= baseline_quick['recall'][idx]
            for idx in preserve_indices
        )
        avoids_recall_collapse = bool(np.all(quick['recall'] + recall_tolerance >= minimum_recall))
        candidate = (
            quick['macro_f1'],
            quick['balanced_accuracy'],
            -quick['fp'][very_long_index],
            multiplier_values.copy(),
            float(threshold),
        )
        if preserves_priority_recall and avoids_recall_collapse:
            relaxed_adjusted_candidates.append(candidate)
            if quick['fp'][very_long_index] <= target_vl_fp:
                adjusted_candidates.append(candidate)

final_adjustment_pool = adjusted_candidates or relaxed_adjusted_candidates
if final_adjustment_pool:
    _, _, _, selected_multiplier_values, selected_very_long_threshold = max(final_adjustment_pool)
else:
    selected_multiplier_values = np.ones(N_CLASSES)
    selected_very_long_threshold = 0.35

selected_class_multipliers = {
    label: float(selected_multiplier_values[idx]) for idx, label in enumerate(CLASS_ORDER)
}

def apply_class_adjustment(probabilities, multipliers, very_long_threshold):
    multiplier_array = np.asarray([multipliers[label] for label in CLASS_ORDER], dtype=float)
    adjusted = probabilities * multiplier_array
    adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)
    predictions = adjusted.argmax(axis=1)
    if very_long_threshold is not None:
        replace_mask = (
            (predictions == very_long_index)
            & (adjusted[:, very_long_index] < very_long_threshold)
        )
        predictions[replace_mask] = adjusted[replace_mask, :very_long_index].argmax(axis=1)
    return adjusted, predictions

adjusted_ensemble_val, adjusted_ensemble_val_predictions = apply_class_adjustment(
    ensemble_val_probabilities,
    selected_class_multipliers,
    selected_very_long_threshold,
)
adjusted_ensemble_metrics = evaluate_probabilities(
    y_val,
    adjusted_ensemble_val,
    adjusted_ensemble_val_predictions,
)


## 9. Validation Selection

Select the highest validation Macro-F1 configuration.


In [188]:
validation_candidates = {
    f"CatBoost [{best_strategies['CatBoost']}]": best_val_metrics['CatBoost'],
    f"RandomForest [{best_strategies['RandomForest']}]": best_val_metrics['RandomForest'],
    f"XGBoost [{best_strategies['XGBoost']}]": best_val_metrics['XGBoost'],
    'SoftVoting': unadjusted_ensemble_metrics,
    'AdjustedSoftVoting': adjusted_ensemble_metrics,
}
validation_comparison = pd.DataFrame([
    compact_metric_row(name, details) for name, details in validation_candidates.items()
]).sort_values(['macro_f1', 'balanced_accuracy'], ascending=False).reset_index(drop=True)

selected_configuration = validation_comparison.iloc[0]['configuration']
frozen_validation_metrics = validation_candidates[selected_configuration]

display(validation_comparison.round(4))


,configuration,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,RandomForest [sqrt_balanced],0.5676,0.5382,0.5317,0.5433
1,XGBoost [none],0.5562,0.5275,0.5286,0.5404
2,CatBoost [balanced],0.5390,0.5322,0.5063,0.5188
3,AdjustedSoftVoting,0.5714,0.5085,0.4994,0.5216
4,SoftVoting,0.5638,0.5001,0.4606,0.4853


## 10. Frozen Test Evaluation

Evaluate the frozen configuration once on the held-out test set.


In [189]:
best_test_probabilities = {
    name: predict_component_proba(best_models[name], X_test) for name in MODEL_NAMES
}

if calibration_retained_for_ensemble:
    ensemble_test_component_probabilities = dict(best_test_probabilities)
    for name, component in calibrated_components.items():
        ensemble_test_component_probabilities[name] = predict_component_proba(component, X_test)
else:
    ensemble_test_component_probabilities = best_test_probabilities

ensemble_test_probabilities = blend_probabilities(
    ensemble_test_component_probabilities,
    selected_ensemble_weights,
)
adjusted_ensemble_test, adjusted_ensemble_test_predictions = apply_class_adjustment(
    ensemble_test_probabilities,
    selected_class_multipliers,
    selected_very_long_threshold,
)

test_candidates = {
    f"CatBoost [{best_strategies['CatBoost']}]": evaluate_probabilities(y_test, best_test_probabilities['CatBoost']),
    f"RandomForest [{best_strategies['RandomForest']}]": evaluate_probabilities(y_test, best_test_probabilities['RandomForest']),
    f"XGBoost [{best_strategies['XGBoost']}]": evaluate_probabilities(y_test, best_test_probabilities['XGBoost']),
    'SoftVoting': evaluate_probabilities(y_test, ensemble_test_probabilities),
    'AdjustedSoftVoting': evaluate_probabilities(
        y_test,
        adjusted_ensemble_test,
        adjusted_ensemble_test_predictions,
    ),
}
test_comparison = pd.DataFrame([
    compact_metric_row(name, details) for name, details in test_candidates.items()
]).sort_values(['macro_f1', 'balanced_accuracy'], ascending=False).reset_index(drop=True)

individual_model = next(
    (name for name in MODEL_NAMES if selected_configuration.startswith(name)),
    None,
)

if individual_model is not None:
    selected_model_name = individual_model
    selected_test_probabilities = best_test_probabilities[individual_model]
    selected_test_predictions = selected_test_probabilities.argmax(axis=1)
    frozen_ensemble_weights = {name: float(name == individual_model) for name in MODEL_NAMES}
    frozen_class_multipliers = {label: 1.0 for label in CLASS_ORDER}
    frozen_very_long_threshold = None
    frozen_calibration_retained = False
    frozen_weight_settings = {individual_model: best_strategies[individual_model]}
else:
    selected_model_name = selected_configuration
    is_adjusted = selected_configuration == 'AdjustedSoftVoting'
    selected_test_probabilities = adjusted_ensemble_test if is_adjusted else ensemble_test_probabilities
    selected_test_predictions = (
        adjusted_ensemble_test_predictions
        if is_adjusted
        else selected_test_probabilities.argmax(axis=1)
    )
    frozen_ensemble_weights = selected_ensemble_weights.copy()
    frozen_class_multipliers = (
        selected_class_multipliers.copy()
        if is_adjusted
        else {label: 1.0 for label in CLASS_ORDER}
    )
    frozen_very_long_threshold = float(selected_very_long_threshold) if is_adjusted else None
    frozen_calibration_retained = calibration_retained_for_ensemble
    frozen_weight_settings = best_strategies.copy()

frozen_test_metrics = evaluate_probabilities(
    y_test,
    selected_test_probabilities,
    selected_test_predictions,
)

selected_test_summary = pd.DataFrame([{
    'selected_configuration': selected_configuration,
    'accuracy': frozen_test_metrics['accuracy'],
    'balanced_accuracy': frozen_test_metrics['balanced_accuracy'],
    'macro_f1': frozen_test_metrics['macro_f1'],
    'weighted_f1': frozen_test_metrics['weighted_f1'],
}])

display(test_comparison.round(4))
display(selected_test_summary.round(4))
display(frozen_test_metrics['per_class'].round(4))
display(frozen_test_metrics['confusion_matrix'])


,configuration,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,XGBoost [none],0.4648,0.4616,0.4108,0.4455
1,CatBoost [balanced],0.4400,0.4601,0.4016,0.4190
2,AdjustedSoftVoting,0.5010,0.4479,0.3887,0.4503
3,RandomForest [sqrt_balanced],0.4495,0.4400,0.3844,0.4231
4,SoftVoting,0.5181,0.4569,0.3717,0.4390


,selected_configuration,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,RandomForest [sqrt_balanced],0.4495,0.44,0.3844,0.4231


,class,precision,recall,f1,FP,FN,support
0,short,0.6452,0.4149,0.5051,55,141,241
1,medium,0.3043,0.2090,0.2478,64,106,134
2,long,0.6364,0.1458,0.2373,4,41,48
3,very_long,0.3783,0.9902,0.5474,166,1,102


,short,medium,long,very_long
short,100,64,1,76
medium,52,28,2,52
long,3,0,7,38
very_long,0,0,1,101


## 11. Deployment Training and Saving

Retrain on all data and save the required deployment outputs.


In [190]:
model_output_dir = OUTPUTS_DIR / 'model_duration_band'
model_output_dir.mkdir(parents=True, exist_ok=True)

predictions_path = model_output_dir / 'model2_duration_band_test_predictions.csv'
model_path = model_output_dir / 'model2_duration_band_inference_bundle.pkl'
handoff_path = model_output_dir / 'model2_v1_duration_band_recommendation_handoff.csv'

probability_columns = [f'prob_{label}' for label in CLASS_ORDER]
predicted_labels = np.asarray([INDEX_TO_CLASS[int(code)] for code in selected_test_predictions])
actual_labels = np.asarray([INDEX_TO_CLASS[int(code)] for code in y_test])
selected_confidence = selected_test_probabilities[
    np.arange(len(selected_test_predictions)),
    selected_test_predictions,
]

predictions_df = pd.DataFrame(selected_test_probabilities, columns=probability_columns)
predictions_df.insert(0, 'actual_duration_band', actual_labels)
predictions_df.insert(1, 'predicted_duration_band', predicted_labels)
predictions_df.insert(2, 'prediction_confidence', selected_confidence)

expected_prediction_columns = [
    'actual_duration_band',
    'predicted_duration_band',
    'prediction_confidence',
    'prob_short',
    'prob_medium',
    'prob_long',
    'prob_very_long',
]
assert predictions_df.columns.tolist() == expected_prediction_columns
assert probability_columns == ['prob_short', 'prob_medium', 'prob_long', 'prob_very_long']
assert np.allclose(predictions_df[probability_columns].sum(axis=1).to_numpy(), 1.0, atol=1e-6)
assert predictions_df['predicted_duration_band'].isin(CLASS_ORDER).all()
predictions_df.to_csv(predictions_path, index=False)

def fit_full_component(model_name, strategy, calibrated):
    if calibrated:
        return fit_calibrated_component(model_name, strategy, X, y)
    return fit_model_pipeline(model_name, strategy, X, y)

final_deployment_models = {}
if selected_model_name in MODEL_NAMES:
    final_deployment_models[selected_model_name] = fit_full_component(
        selected_model_name,
        best_strategies[selected_model_name],
        False,
    )
else:
    for name in MODEL_NAMES:
        if frozen_ensemble_weights[name] > 0:
            final_deployment_models[name] = fit_full_component(
                name,
                best_strategies[name],
                frozen_calibration_retained and name != 'CatBoost',
            )

preprocessing_objects = {}
calibration_objects = {}
encoded_feature_columns = {}
for name, component in final_deployment_models.items():
    if isinstance(component, dict) and component.get('calibrated', False):
        preprocessor = component['preprocessor']
        calibration_objects[name] = component['model']
    else:
        preprocessor = component.named_steps['preprocessor']
    preprocessing_objects[name] = preprocessor
    encoded_feature_columns[name] = preprocessor.get_feature_names_out().tolist()

def serializable_metrics(details):
    return {
        'accuracy': float(details['accuracy']),
        'balanced_accuracy': float(details['balanced_accuracy']),
        'macro_f1': float(details['macro_f1']),
        'weighted_f1': float(details['weighted_f1']),
        'log_loss': float(details['log_loss']),
        'per_class': details['per_class'].to_dict(orient='records'),
        'confusion_matrix': details['confusion_matrix'].to_numpy().astype(int).tolist(),
    }

model_artifact = {
    'models': final_deployment_models,
    'preprocessing_objects': preprocessing_objects,
    'selected_configuration': selected_configuration,
    'selected_model_type': selected_model_name,
    'class_weight_setting': frozen_weight_settings,
    'class_order': CLASS_ORDER.copy(),
    'ensemble_weights': {k: float(v) for k, v in frozen_ensemble_weights.items()},
    'class_multipliers': frozen_class_multipliers.copy(),
    'very_long_threshold': frozen_very_long_threshold,
    'calibration_retained': bool(frozen_calibration_retained),
    'calibration_objects': calibration_objects or None,
    'model_input_columns': feature_cols.copy(),
    'numeric_input_columns': numeric_cols.copy(),
    'categorical_input_columns': categorical_cols.copy(),
    'encoded_feature_columns': encoded_feature_columns,
    'validation_metrics': serializable_metrics(frozen_validation_metrics),
    'test_metrics': serializable_metrics(frozen_test_metrics),
    'deployment_training_mode': 'full_dataset_training',
    'model_selection_mode': SPLIT_METHOD,
}
joblib.dump(model_artifact, model_path)

# Preserve the notebook's required downstream recommendation handoff and its exact schema.
recommendation_handoff_df = model_df.loc[X_test.index].reset_index(drop=True).copy()
recommendation_handoff_df = pd.concat(
    [recommendation_handoff_df, predictions_df.drop(columns=['actual_duration_band']).reset_index(drop=True)],
    axis=1,
)
final_handoff_columns = [
    'id', '_source_row', 'prediction_datetime', 'start_datetime', 'latitude', 'longitude',
    'location_grid', 'distance_band', 'address', 'event_type', 'event_cause', 'veh_type',
    'corridor', 'police_station', 'zone', 'junction', 'start_hour', 'start_dayofweek',
    'start_month_number', 'is_weekend', 'is_morning_peak', 'is_evening_peak',
    'is_peak_hour', 'is_night', 'peak_period', 'report_lag_minutes_clipped',
    'distance_to_city_center_km', 'description', 'is_planned_event',
    'is_public_or_vip_event', 'is_breakdown_event', 'is_accident_event',
    'is_weather_or_visibility_event', 'is_road_condition_event', 'is_heavy_vehicle',
    'has_blocked_word', 'has_jam_word', 'has_lane_word', 'has_full_blockage_phrase',
    'has_partial_blockage_phrase', 'has_diversion_word', 'has_severity_word',
    'has_tree_fall_word', 'target_road_closure', 'valid_duration_label', 'duration_band',
    'road_closure_probability', 'predicted_label',
    'road_closure_probability_is_history_fallback', 'predicted_duration_band',
    'prediction_confidence', 'prob_short', 'prob_medium', 'prob_long', 'prob_very_long',
]
recommendation_handoff_df = recommendation_handoff_df.reindex(columns=final_handoff_columns)
recommendation_handoff_df.to_csv(handoff_path, index=False)

display(predictions_df.head(10).round(6))


,actual_duration_band,predicted_duration_band,prediction_confidence,prob_short,prob_medium,prob_long,prob_very_long
0,short,medium,0.516152,0.470557,0.516152,0.004183,0.009108
1,medium,short,0.502558,0.502558,0.488220,0.004722,0.004500
2,short,medium,0.535141,0.459026,0.535141,0.002222,0.003611
3,short,very_long,0.476809,0.149960,0.137156,0.236076,0.476809
4,short,very_long,0.376972,0.231456,0.184527,0.207045,0.376972
5,short,medium,0.554321,0.431578,0.554321,0.007598,0.006504
6,short,medium,0.585109,0.405169,0.585109,0.004444,0.005278
7,very_long,very_long,0.595867,0.126638,0.105961,0.171534,0.595867
8,long,very_long,0.408637,0.194994,0.087900,0.308469,0.408637
9,short,short,0.524238,0.524238,0.465345,0.005139,0.005278


## 12. Feature Importance

Display the top deployment features without writing another CSV.


In [191]:
feature_importance_frames = []

for model_name, component in final_deployment_models.items():
    if isinstance(component, dict) and component.get('calibrated', False):
        preprocessor = component['preprocessor']
        estimators = [
            calibrated.estimator
            for calibrated in component['model'].calibrated_classifiers_
            if hasattr(calibrated.estimator, 'feature_importances_')
        ]
    else:
        preprocessor = component.named_steps['preprocessor']
        estimator = component.named_steps['model']
        estimators = [estimator] if hasattr(estimator, 'feature_importances_') else []

    if not estimators:
        continue

    feature_names = np.asarray(preprocessor.get_feature_names_out(), dtype=object)
    importance_values = np.mean(
        [np.asarray(estimator.feature_importances_, dtype=float) for estimator in estimators],
        axis=0,
    )
    assert len(feature_names) == len(importance_values)

    model_importance = pd.DataFrame({
        'model': model_name,
        'feature': feature_names,
        'importance': importance_values,
    }).sort_values('importance', ascending=False)
    feature_importance_frames.append(model_importance.head(20))

feature_importance_check = pd.concat(feature_importance_frames, ignore_index=True)
display(feature_importance_check.round({'importance': 6}))


,model,feature,importance
0,RandomForest,numeric__past_duration_rate_short_event_cause,0.020162
1,RandomForest,numeric__past_duration_q75_event_cause,0.019031
2,RandomForest,numeric__past_duration_rate_long_event_cause,0.017878
3,RandomForest,numeric__past_duration_median_event_cause,0.016610
4,RandomForest,numeric__past_duration_rate_very_long_event_cause,0.016055
5,RandomForest,numeric__past_duration_rate_medium_event_cause,0.015589
6,RandomForest,numeric__past_duration_rate_very_long_event_ca...,0.015038
7,RandomForest,numeric__past_duration_q25_event_cause,0.014797
8,RandomForest,categorical__veh_type___MISSING__,0.013333
9,RandomForest,numeric__past_duration_median_event_cause_corr...,0.013062


## 13. Output Checks

Verify the required bundle, predictions, and downstream handoff.


In [192]:
required_paths = [predictions_path, model_path, handoff_path]
for required_path in required_paths:
    print(f'{required_path}: {required_path.exists()}')
assert all(path.exists() for path in required_paths)


D:\Python\Gridlock\Phase 2\theme 2\outputs\model_duration_band\model2_duration_band_test_predictions.csv: True
D:\Python\Gridlock\Phase 2\theme 2\outputs\model_duration_band\model2_duration_band_inference_bundle.pkl: True
D:\Python\Gridlock\Phase 2\theme 2\outputs\model_duration_band\model2_v1_duration_band_recommendation_handoff.csv: True


## 14. Inference

Apply the saved preprocessing and frozen decision logic to new rows.


In [193]:
def predict_duration_band(row_or_df, bundle=model_artifact):
    event_df = row_or_df.to_frame().T if isinstance(row_or_df, pd.Series) else row_or_df.copy()
    event_X = event_df.reindex(columns=bundle['model_input_columns'])

    component_probabilities = {}
    for name, component in bundle['models'].items():
        component_probabilities[name] = predict_component_proba(component, event_X)

    if bundle['selected_model_type'] in MODEL_NAMES:
        probabilities = component_probabilities[bundle['selected_model_type']]
    else:
        probabilities = sum(
            bundle['ensemble_weights'][name] * component_probabilities[name]
            for name in component_probabilities
        )
        probabilities = probabilities / probabilities.sum(axis=1, keepdims=True)

    probabilities, predictions = apply_class_adjustment(
        probabilities,
        bundle['class_multipliers'],
        bundle['very_long_threshold'],
    )
    labels = [bundle['class_order'][int(code)] for code in predictions]
    confidence = probabilities[np.arange(len(predictions)), predictions]
    result = pd.DataFrame(
        probabilities,
        columns=[f'prob_{label}' for label in bundle['class_order']],
    )
    result.insert(0, 'predicted_duration_band', labels)
    result.insert(1, 'prediction_confidence', confidence)
    return result


## 15. Summary

Show the final configuration, metrics, and saved paths.


In [194]:
project_validation_summary = pd.DataFrame([
    ('dataset_split_method', SPLIT_METHOD),
    ('selected_final_configuration', selected_configuration),
    ('selected_ensemble_weights', frozen_ensemble_weights),
    ('selected_class_multipliers', frozen_class_multipliers),
    ('selected_very_long_threshold', frozen_very_long_threshold),
    ('calibration_retained', frozen_calibration_retained),
    ('validation_macro_f1', round(frozen_validation_metrics['macro_f1'], 4)),
    ('validation_balanced_accuracy', round(frozen_validation_metrics['balanced_accuracy'], 4)),
    ('test_macro_f1', round(frozen_test_metrics['macro_f1'], 4)),
    ('test_balanced_accuracy', round(frozen_test_metrics['balanced_accuracy'], 4)),
    ('deployment_bundle', str(model_path)),
    ('test_prediction_csv', str(predictions_path)),
    ('downstream_handoff_csv', str(handoff_path)),
    ('deployment_retrained_on_full_dataset', True),
], columns=['item', 'value'])
display(project_validation_summary)


,item,value
0,dataset_split_method,chronological_train_validation_test_split
1,selected_final_configuration,RandomForest [sqrt_balanced]
2,selected_ensemble_weights,"{'CatBoost': 0.0, 'RandomForest': 1.0, 'XGBoos..."
3,selected_class_multipliers,"{'short': 1.0, 'medium': 1.0, 'long': 1.0, 've..."
4,selected_very_long_threshold,None
5,calibration_retained,False
6,validation_macro_f1,0.5317
7,validation_balanced_accuracy,0.5382
8,test_macro_f1,0.3844
9,test_balanced_accuracy,0.44
